In [0]:
# OBJECTIF : Nettoyage et structuration des données hospitalières
# CONFIGURATION DATABRICKS
# spark.conf.set("spark.sql.adaptive.enabled", "true")
# spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
#print("Configuration Spark optimisée")
# print(f"Version Spark : {spark.version}")
#print(f"Databricks Runtime : {sc.version}")

In [0]:
# ========================================
# IMPORTS
# ========================================
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window
from delta.tables import DeltaTable
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns
print("Imports réussis")

Imports réussis


In [0]:
# ========================================
# CHARGEMENT DONNÉES BRUTES
# ========================================
file_path = "/Volumes/workspace/default/amdy_hopitalisation_data/hospital_readmissions_dataset.csv"
# Schéma explicite (CRITICAL : ne jamais laisser l'inférence deviner)
schema = StructType([
    StructField("patient_id", StringType(), False),
    StructField("age", IntegerType(), True),
    StructField("genre", StringType(), True),
    StructField("diabete", StringType(), True),
    StructField("hypertension", StringType(), True),
    StructField("insuffisance_cardiaque", StringType(), True),
    StructField("maladie_renale", StringType(), True),
    StructField("cancer", StringType(), True),
    StructField("depression", StringType(), True),
    StructField("nb_comorbidites", IntegerType(), True),
    StructField("charlson_index", IntegerType(), True),
    StructField("diagnostic_principal", StringType(), True),
    StructField("duree_sejour_jours", IntegerType(), True),
    StructField("admission_urgence", StringType(), True),
    StructField("service", StringType(), True),
    StructField("date_admission", StringType(), True),
    StructField("date_sortie", StringType(), True),
    StructField("saison", StringType(), True),
    StructField("proba_readmission", DoubleType(), True), 
    StructField("readmission_30j", StringType(), False),   # VARIABLE CIBLE
    StructField("nb_admissions_12mois", IntegerType(), True),
    StructField("nb_medicaments", IntegerType(), True),
    StructField("score_gravite_admission", IntegerType(), True),
    StructField("support_social", StringType(), True),
    StructField("distance_hopital_km", IntegerType(), True)
])
# Chargement avec gestion erreurs
df_raw = (spark.read
          .format("csv")
          .option("header", "true")
          .option("delimiter", "\t")  # ATTENTION : vérifier si TAB ou virgule
          .option("quote", "\"")
          .option("escape", "\"")
          .option("mode", "DROPMALFORMED")  # Drop lignes mal formées
          .schema(schema)
          .load(file_path))

# Vérification
print(f"Lignes chargées : {df_raw.count():,}")
print(f"Colonnes : {len(df_raw.columns)}")
df_raw.printSchema()

Lignes chargées : 69,750
Colonnes : 25
root
 |-- patient_id: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- genre: string (nullable = true)
 |-- diabete: string (nullable = true)
 |-- hypertension: string (nullable = true)
 |-- insuffisance_cardiaque: string (nullable = true)
 |-- maladie_renale: string (nullable = true)
 |-- cancer: string (nullable = true)
 |-- depression: string (nullable = true)
 |-- nb_comorbidites: integer (nullable = true)
 |-- charlson_index: integer (nullable = true)
 |-- diagnostic_principal: string (nullable = true)
 |-- duree_sejour_jours: integer (nullable = true)
 |-- admission_urgence: string (nullable = true)
 |-- service: string (nullable = true)
 |-- date_admission: string (nullable = true)
 |-- date_sortie: string (nullable = true)
 |-- saison: string (nullable = true)
 |-- proba_readmission: double (nullable = true)
 |-- readmission_30j: string (nullable = true)
 |-- nb_admissions_12mois: integer (nullable = true)
 |-- nb_medicame

In [0]:
# ========================================
# AUDIT QUALITÉ DES DONNÉES - VERSION PRODUCTION
# ========================================
def audit_data_quality(df):
    """Génère un rapport de qualité avec gestion robuste des types"""
    
    # DIAGNOSTIC DES TYPES PROBLÉMATIQUES
    print("Diagnostic des types problématiques :")
    for field in df.schema.fields:
        if str(field.dataType) in ['LongType', 'IntegerType', 'DoubleType']:
            print(f"   Colonne numérique detectee : {field.name} ({field.dataType})")
            # Verifier si cette colonne contient des chaines vides ou espaces
            empty_count = df.filter(F.col(field.name).cast("string").rlike("^\\s*$")).count()
            if empty_count > 0:
                print(f"      Valeurs vides ou espaces trouvees : {empty_count} => SOURCE DU PROBLEME !")

    # NETTOYAGE GLOBAL AVANT TRAITEMENT
    # Remplacer chaines vides, espaces, "NA", "null" par NULL dans TOUTES les colonnes
    for col_name in df.columns:
        df = df.withColumn(
            col_name,
            F.when(F.col(col_name).cast("string").rlike("^\\s*$|^(NA|null|None)$"), 
                   F.lit(None))
             .otherwise(F.col(col_name))
        )
    
    # FORCER patient_id en string (securite supplementaire)
    df = df.withColumn("patient_id", F.col("patient_id").cast("string"))
    
    # CONVERTIR LES COLONNES NUMERIQUES PROBLEMATIQUES
    # Liste des colonnes de type numerique dans votre schema
    numeric_cols = ["age", "nb_comorbidites", "charlson_index", "duree_sejour_jours", 
                    "nb_admissions_12mois", "nb_medicaments", "score_gravite_admission", 
                    "distance_hopital_km", "proba_readmission"]
    
    for num_col in numeric_cols:
        if num_col in df.columns:
            # Utiliser try_cast pour convertir en double (plus permissif)
            df = df.withColumn(num_col, F.expr(f"try_cast({num_col} as double)"))

    total_rows = df.count()
    print(f"\n{'='*60}")
    print(f"RAPPORT DE QUALITE - {total_rows:,} lignes")
    print(f"{'='*60}\n")

    # COMPTAGE DES VALEURS MANQUANTES (APPROCHE SAFE)
    print("VALEURS MANQUANTES :")
    missing_report = []
    for col in df.columns:
        # Utiliser cast(string) pour eviter les conversions implicites dangereuses
        null_count = df.filter(
            F.col(col).isNull() | 
            F.expr(f"cast({col} as string) = ''")
        ).count()
        
        null_pct = (null_count / total_rows) * 100
        missing_report.append({
            'colonne': col,
            'nb_nulls': null_count,
            'pct_nulls': round(null_pct, 2)
        })
    
    missing_df = pd.DataFrame(missing_report).sort_values('pct_nulls', ascending=False)
    print(missing_df.to_string(index=False))

    # DOUBLONS
    print(f"\nDOUBLONS :")
    duplicates = df.count() - df.dropDuplicates().count()
    print(f"   Lignes dupliquees (exactes) : {duplicates}")
    
    # DOUBLONS PATIENT (avec try_cast securise)
    duplicates_patient = df.groupBy(
        F.expr("try_cast(patient_id as string)").alias("patient_id_safe")
    ).count().filter(F.col("count") > 1).count()
    print(f"   Patients avec plusieurs enregistrements : {duplicates_patient}")

    # COHERENCE TEMPORELLE
    print(f"\nCOHERENCE TEMPORELLE :")
    df_temp = df.withColumn("date_adm_parsed", F.to_date(F.col("date_admission"), "dd/MM/yyyy HH:mm")) \
                .withColumn("date_sort_parsed", F.to_date(F.col("date_sortie"), "dd/MM/yyyy HH:mm"))
    invalid_dates = df_temp.filter(F.col("date_sort_parsed") < F.col("date_adm_parsed")).count()
    print(f"   Dates de sortie avant admission : {invalid_dates}")
    
    return missing_df

# ETAPE 1 : VERIFICATION PREALABLE
print("=== VERIFICATION DU SCHEMA ===")
df_raw.printSchema()
print("\nApercu des donnees :")
display(df_raw.limit(5))

# ETAPE 2 : VERIFICATION DES VALEURS CRITIQUES
print("\nValeurs vides ou espaces dans les colonnes numeriques :")
numeric_cols = ["age", "nb_comorbidites", "charlson_index", "duree_sejour_jours", 
                "nb_admissions_12mois", "nb_medicaments", "score_gravite_admission", 
                "distance_hopital_km", "proba_readmission"]

for num_col in numeric_cols:
    if num_col in df_raw.columns:
        count_empty = df_raw.filter(F.col(num_col).cast("string").rlike("^\\s*$")).count()
        if count_empty > 0:
            print(f"   {num_col}: {count_empty} valeurs vides")

# ETAPE 3 : EXECUTION DE L'AUDIT
missing_summary = audit_data_quality(df_raw)

=== VERIFICATION DU SCHEMA ===
root
 |-- patient_id: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- genre: string (nullable = true)
 |-- diabete: string (nullable = true)
 |-- hypertension: string (nullable = true)
 |-- insuffisance_cardiaque: string (nullable = true)
 |-- maladie_renale: string (nullable = true)
 |-- cancer: string (nullable = true)
 |-- depression: string (nullable = true)
 |-- nb_comorbidites: integer (nullable = true)
 |-- charlson_index: integer (nullable = true)
 |-- diagnostic_principal: string (nullable = true)
 |-- duree_sejour_jours: integer (nullable = true)
 |-- admission_urgence: string (nullable = true)
 |-- service: string (nullable = true)
 |-- date_admission: string (nullable = true)
 |-- date_sortie: string (nullable = true)
 |-- saison: string (nullable = true)
 |-- proba_readmission: double (nullable = true)
 |-- readmission_30j: string (nullable = true)
 |-- nb_admissions_12mois: integer (nullable = true)
 |-- nb_medicaments: int

patient_id,age,genre,diabete,hypertension,insuffisance_cardiaque,maladie_renale,cancer,depression,nb_comorbidites,charlson_index,diagnostic_principal,duree_sejour_jours,admission_urgence,service,date_admission,date_sortie,saison,proba_readmission,readmission_30j,nb_admissions_12mois,nb_medicaments,score_gravite_admission,support_social,distance_hopital_km



Valeurs vides ou espaces dans les colonnes numeriques :
Diagnostic des types problématiques :

RAPPORT DE QUALITE - 69,750 lignes

VALEURS MANQUANTES :
                colonne  nb_nulls  pct_nulls
     duree_sejour_jours     69750      100.0
      admission_urgence     69750      100.0
         support_social     69750      100.0
score_gravite_admission     69750      100.0
         nb_medicaments     69750      100.0
   nb_admissions_12mois     69750      100.0
        readmission_30j     69750      100.0
      proba_readmission     69750      100.0
                 saison     69750      100.0
            date_sortie     69750      100.0
         date_admission     69750      100.0
                service     69750      100.0
    distance_hopital_km     69750      100.0
                    age     69750      100.0
   diagnostic_principal     69750      100.0
         charlson_index     69750      100.0
        nb_comorbidites     69750      100.0
             depression     69750    

In [0]:
# ========================================
# SUPPRESSION DATA LEAKAGE
# ========================================
# pourquoi : La colonne 'proba_readmission' contient des prédictions
#       Elle ne devrait PAS exister dans les données brutes
#       C'est une fuite de données qui invaliderait le modèle
print("Suppression colonne 'proba_readmission' (data leakage)")
df_clean = df_raw.drop("proba_readmission")
print(f"Colonnes restantes : {len(df_clean.columns)}")

Suppression colonne 'proba_readmission' (data leakage)
Colonnes restantes : 24


In [0]:
# ========================================
# CONVERSION BOOLÉENS
# ========================================
# WHY : Les colonnes booléennes sont en texte "VRAI"/"FAUX"
#       Conversion vers vrais booléens pour efficacité mémoire et calculs
bool_columns = [
    "diabete", "hypertension", "insuffisance_cardiaque",
    "maladie_renale", "cancer", "depression", "admission_urgence",
    "readmission_30j"  # CIBLE
]
print(f" Conversion de {len(bool_columns)} colonnes booléennes...")
for col in bool_columns:
    df_clean = df_clean.withColumn(
        col,
        F.when(F.upper(F.col(col)) == "VRAI", True)
         .when(F.upper(F.col(col)) == "FAUX", False)
         .otherwise(None)  # Gestion valeurs inattendues
    )
# Vérification
print("\n Échantillon après conversion :")
df_clean.select(bool_columns[:3] + ["readmission_30j"]).show(5)

 Conversion de 8 colonnes booléennes...

 Échantillon après conversion :
+-------+------------+----------------------+---------------+
|diabete|hypertension|insuffisance_cardiaque|readmission_30j|
+-------+------------+----------------------+---------------+
|   NULL|        NULL|                  NULL|           NULL|
|   NULL|        NULL|                  NULL|           NULL|
|   NULL|        NULL|                  NULL|           NULL|
|   NULL|        NULL|                  NULL|           NULL|
|   NULL|        NULL|                  NULL|           NULL|
+-------+------------+----------------------+---------------+
only showing top 5 rows


In [0]:
# ========================================
# TRAITEMENT DATES
# ========================================
# WHY : Dates en string → conversion en timestamps
#       Calcul durée séjour réelle vs déclarée
#       Détection incohérences temporelles
print("Parsing des dates...")
df_clean = (df_clean
    .withColumn("date_admission_ts", 
                F.to_timestamp(F.col("date_admission"), "dd/MM/yyyy HH:mm"))
    .withColumn("date_sortie_ts", 
                F.to_timestamp(F.col("date_sortie"), "dd/MM/yyyy HH:mm"))
    .withColumn("duree_sejour_calculee", 
                F.datediff(F.col("date_sortie_ts"), F.col("date_admission_ts")))
)
# Détection incohérences
print("\n Détection incohérences temporelles :")
invalid_dates = df_clean.filter(
    F.col("date_sortie_ts") < F.col("date_admission_ts")
).count()
print(f" Sorties avant admission : {invalid_dates}")
# Comparaison durée déclarée vs calculée
discrepancy = df_clean.filter(
    F.col("duree_sejour_jours") != F.col("duree_sejour_calculee")
).count()
print(f" Écarts durée séjour déclarée/calculée : {discrepancy}")
# Afficher échantillon des écarts
if discrepancy > 0:
    print("\n Échantillon écarts :")
    df_clean.filter(
        F.col("duree_sejour_jours") != F.col("duree_sejour_calculee")
    ).select("patient_id", "duree_sejour_jours", "duree_sejour_calculee") \
     .show(10)

Parsing des dates...

 Détection incohérences temporelles :
 Sorties avant admission : 0
 Écarts durée séjour déclarée/calculée : 0


In [0]:
# ========================================
# STRATÉGIE VALEURS MANQUANTES
# ========================================
# WHY : Décisions basées sur le contexte métier
#       Pas d'imputation aveugle
print("STRATÉGIE DE GESTION DES NULLS :\n")
# 1. Variables critiques : DROP si null
critical_vars = ["patient_id", "age", "genre", "readmission_30j", "date_admission_ts"]
print(f" Variables critiques (DROP si null) : {critical_vars}")
rows_before = df_clean.count()
df_clean = df_clean.dropna(subset=critical_vars)
rows_after = df_clean.count()
print(f"lignes supprimées : {rows_before - rows_after}")
# 2. Comorbidités : NULL → False (hypothèse conservatrice)
comorbidity_cols = ["diabete", "hypertension", "insuffisance_cardiaque",
                    "maladie_renale", "cancer", "depression"]
print(f"\n Comorbidités (NULL → False) : {len(comorbidity_cols)} colonnes")
for col in comorbidity_cols:
    df_clean = df_clean.withColumn(col, F.coalesce(F.col(col), F.lit(False)))
# 3. Variables numériques : imputation par médiane (par service)
numeric_cols_to_impute = ["nb_medicaments", "score_gravite_admission", "distance_hopital_km"]
print(f"\n Numériques (imputation médiane par service) : {numeric_cols_to_impute}")
for col in numeric_cols_to_impute:
    # Calcul médiane par service
    median_by_service = df_clean.groupBy("service").agg(
        F.expr(f"percentile_approx({col}, 0.5)").alias(f"median_{col}")
    )
    # Join et imputation
    df_clean = df_clean.join(median_by_service, "service", "left")
    df_clean = df_clean.withColumn(
        col,
        F.coalesce(F.col(col), F.col(f"median_{col}"))
    ).drop(f"median_{col}")
print("\n Imputation terminée")

STRATÉGIE DE GESTION DES NULLS :

 Variables critiques (DROP si null) : ['patient_id', 'age', 'genre', 'readmission_30j', 'date_admission_ts']
lignes supprimées : 69750

 Comorbidités (NULL → False) : 6 colonnes

 Numériques (imputation médiane par service) : ['nb_medicaments', 'score_gravite_admission', 'distance_hopital_km']

 Imputation terminée


In [0]:
# ========================================
# DÉTECTION OUTLIERS - VERSION ROBUSTE
# ========================================
print(" OUTLIERS (méthode IQR)...\n")
outlier_cols = ["age", "duree_sejour_calculee", "nb_medicaments", 
                "score_gravite_admission", "distance_hopital_km"]
def detect_outliers_iqr(df, col_name):
    """Détecte outliers avec gestion d'erreurs robuste"""
    # Vérification 1 : Colonne existe
    if col_name not in df.columns:
        print(f"   ERREUR: Colonne {col_name} introuvable")
        return None, None, 0
    # Vérification 2 : Colonne numérique
    col_type = df.schema[col_name].dataType
    if not str(col_type).startswith(('Integer', 'Long', 'Double', 'Float')):
        print(f"   ERREUR: Colonne {col_name} non numérique (type: {col_type})")
        return None, None, 0
    # Vérification 3 : Assez de données non-NULL
    valid_count = df.filter(F.col(col_name).isNotNull()).count()
    if valid_count < 2:
        print(f"   ERREUR: {col_name} a {valid_count} valeur(s) valide(s) (minimum 2 requis)")
        return None, None, 0
    # Calcul des quantiles avec try-except
    try:
        quantiles = df.approxQuantile(col_name, [0.25, 0.75], 0.01)
    except Exception as e:
        print(f"   ERREUR: Calcul quantiles échoué: {e}")
        return None, None, 0
    # Vérification 4 : Quantiles valides
    if len(quantiles) < 2 or None in quantiles:
        print(f"   ERREUR: Quantiles invalides pour {col_name}: {quantiles}")
        return None, None, 0
    Q1, Q3 = quantiles[0], quantiles[1]
    # Vérification 5 : Cas Q1=Q3
    if Q1 == Q3:
        print(f"   INFO: {col_name} a Q1=Q3={Q1} (toutes valeurs identiques)")
        IQR = 1.0  # Valeur par défaut
    else:
        IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = df.filter(
        (F.col(col_name) < lower_bound) | (F.col(col_name) > upper_bound)
    ).count()
    return lower_bound, upper_bound, outliers
# RAPPORT DES OUTLIERS
outlier_report = []
print("Calcul des outliers par colonne :")
for col in outlier_cols:
    print(f"  Traitement: {col}")
    lower, upper, count = detect_outliers_iqr(df_clean, col)
    if lower is None:  # Erreur détectée
        print(f"    -> Colonne ignorée\n")
        continue
    total_rows = df_clean.count()
    outlier_report.append({
        'variable': col,
        'borne_inf': round(lower, 2),
        'borne_sup': round(upper, 2),
        'nb_outliers': count,
        'pct': round((count / total_rows) * 100, 2)
    })
outlier_df = pd.DataFrame(outlier_report)
print("\n RAPPORT FINAL DES OUTLIERS :")
print(outlier_df.to_string(index=False))

 OUTLIERS (méthode IQR)...

Calcul des outliers par colonne :
  Traitement: age
   ERREUR: age a 0 valeur(s) valide(s) (minimum 2 requis)
    -> Colonne ignorée

  Traitement: duree_sejour_calculee
   ERREUR: duree_sejour_calculee a 0 valeur(s) valide(s) (minimum 2 requis)
    -> Colonne ignorée

  Traitement: nb_medicaments
   ERREUR: nb_medicaments a 0 valeur(s) valide(s) (minimum 2 requis)
    -> Colonne ignorée

  Traitement: score_gravite_admission
   ERREUR: score_gravite_admission a 0 valeur(s) valide(s) (minimum 2 requis)
    -> Colonne ignorée

  Traitement: distance_hopital_km
   ERREUR: distance_hopital_km a 0 valeur(s) valide(s) (minimum 2 requis)
    -> Colonne ignorée


 RAPPORT FINAL DES OUTLIERS :
Empty DataFrame
Columns: []
Index: []


In [0]:
# ========================================
# GESTION DOUBLONS
# ========================================
# WHY : Un patient peut avoir plusieurs admissions (légitime)
#       Mais des lignes 100% identiques = erreur de saisie
print(" DÉTECTION DOUBLONS...\n")
# 1. Doublons exacts (toutes colonnes identiques)
print(" Doublons exacts :")
df_dedup = df_clean.dropDuplicates()
duplicates_exact = df_clean.count() - df_dedup.count()
print(f"   Lignes dupliquées supprimées : {duplicates_exact}")
df_clean = df_dedup
# 2. Patients avec plusieurs admissions (LÉGITIME)
print("\n Patients avec admissions multiples (LÉGITIME) :")
multi_admissions = (df_clean
    .groupBy("patient_id")
    .agg(F.count("*").alias("nb_admissions_patient"))
    .filter("nb_admissions_patient > 1")
    .count())
print(f"   Patients concernés : {multi_admissions}")
# 3. Doublons suspects (même patient, même date, diagnostic différent)
print("\n Doublons suspects (même patient + même date) :")
window_spec = Window.partitionBy("patient_id", "date_admission_ts").orderBy("date_sortie_ts")
df_clean = df_clean.withColumn("admission_rank", F.row_number().over(window_spec))
suspects = df_clean.filter("admission_rank > 1").count()
print(f"   Enregistrements suspects détectés : {suspects}")
if suspects > 0:
    print(" DÉCISION : Garder le premier enregistrement par patient+date")
    df_clean = df_clean.filter("admission_rank = 1").drop("admission_rank")
else:
    df_clean = df_clean.drop("admission_rank")
print(f"\n Dataset final après dédoublonnage : {df_clean.count():,} lignes")

 DÉTECTION DOUBLONS...

 Doublons exacts :
   Lignes dupliquées supprimées : 0

 Patients avec admissions multiples (LÉGITIME) :
   Patients concernés : 0

 Doublons suspects (même patient + même date) :
   Enregistrements suspects détectés : 0

 Dataset final après dédoublonnage : 0 lignes


In [0]:
# ========================================
# VALIDATION RÈGLES MÉTIER
# ========================================
# WHY : Détection anomalies domaine médical
print("VALIDATION RÈGLES MÉTIER...\n")
# Règle 1 : Âge cohérent (18-120 ans pour ce dataset)
invalid_age = df_clean.filter((F.col("age") < 18) | (F.col("age") > 120)).count()
print(f" Âges invalides (<18 ou >120) : {invalid_age}")
if invalid_age > 0:
    df_clean = df_clean.filter((F.col("age") >= 18) & (F.col("age") <= 120))
# Règle 2 : Durée séjour positive
invalid_stay = df_clean.filter(F.col("duree_sejour_calculee") < 0).count()
print(f" Durées séjour négatives : {invalid_stay}")
if invalid_stay > 0:
    df_clean = df_clean.filter(F.col("duree_sejour_calculee") >= 0)
# Règle 3 : Charlson Index cohérent avec comorbidités
print(f"\n Cohérence Charlson Index vs nb_comorbidites :")
incoherent = df_clean.filter(
    (F.col("charlson_index") == 0) & (F.col("nb_comorbidites") > 0)
).count()
print(f"Cas incohérents détectés : {incoherent}")
# Règle 4 : Support social doit être catégorie valide
valid_support = ["Faible", "Moyen", "Élevé"]
invalid_support = df_clean.filter(~F.col("support_social").isin(valid_support)).count()
print(f"\n Support social invalide : {invalid_support}")
if invalid_support > 0:
    # Imputation mode
    mode_support = df_clean.groupBy("support_social").count() \
                           .orderBy(F.desc("count")).first()["support_social"]
    df_clean = df_clean.withColumn(
        "support_social",
        F.when(F.col("support_social").isin(valid_support), F.col("support_social"))
         .otherwise(mode_support)
    )
print(f"\n Dataset après validation métier : {df_clean.count():,} lignes")

VALIDATION RÈGLES MÉTIER...

 Âges invalides (<18 ou >120) : 0
 Durées séjour négatives : 0

 Cohérence Charlson Index vs nb_comorbidites :
Cas incohérents détectés : 0

 Support social invalide : 0

 Dataset après validation métier : 0 lignes


In [0]:
# ========================================
# CRÉATION VARIABLES DÉRIVÉES
# ========================================
# WHY : Features engineering de base durant le cleaning
print("CRÉATION VARIABLES DÉRIVÉES...\n")
df_clean = (df_clean
    # 1. Catégories d'âge
    .withColumn("categorie_age",
        F.when(F.col("age") < 40, "Jeune")
         .when(F.col("age") < 65, "Adulte")
         .otherwise("Senior"))
    # 2. Catégories durée séjour
    .withColumn("categorie_sejour",
        F.when(F.col("duree_sejour_calculee") <= 2, "Court")
         .when(F.col("duree_sejour_calculee") <= 7, "Moyen")
         .otherwise("Long"))
    # 3. Flag polypathologie
    .withColumn("polypathologie", F.col("nb_comorbidites") >= 3)
    # 4. Extraction features temporelles
    .withColumn("annee_admission", F.year("date_admission_ts"))
    .withColumn("mois_admission", F.month("date_admission_ts"))
    .withColumn("jour_semaine", F.dayofweek("date_admission_ts"))
    .withColumn("est_weekend", F.col("jour_semaine").isin([1, 7]))  # 1=Dimanche, 7=Samedi
    # 5. Ratio médicaments / comorbidités
    .withColumn("ratio_medic_comorb",
        F.when(F.col("nb_comorbidites") > 0,
               F.col("nb_medicaments") / F.col("nb_comorbidites"))
         .otherwise(F.col("nb_medicaments")))
)
print("Variables dérivées créées :")
print("categorie_age, categorie_sejour")
print("polypathologie (bool)")
print("annee/mois/jour_semaine/est_weekend")
print("ratio_medic_comorb")

CRÉATION VARIABLES DÉRIVÉES...

Variables dérivées créées :
categorie_age, categorie_sejour
polypathologie (bool)
annee/mois/jour_semaine/est_weekend
ratio_medic_comorb


In [0]:
# ========================================
# SAUVEGARDE DELTA LAKE
# ========================================
# WHY : Format optimisé pour Databricks avec versioning
print(" CRÉATION TABLES DELTA LAKE...\n")
# Définir paths
catalog = "workspace"  # Adapter selon ton setup
schema_name = "default"  # Adapter
table_name = "hospital_readmissions_clean"
full_table_name = f"{catalog}.{schema_name}.{table_name}"
# Sauvegarder en Delta avec partitionnement
print(f" Écriture table : {full_table_name}")
(df_clean.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("annee_admission", "mois_admission")  # Partitionnement temporel
    .saveAsTable(full_table_name))
print(f" Table Delta créée : {full_table_name}")
# Statistiques finales
final_count = spark.table(full_table_name).count()
print(f"\n STATISTIQUES FINALES :")
print(f"   Lignes : {final_count:,}")
print(f"   Colonnes : {len(df_clean.columns)}")
print(f"   Partitions : annee_admission × mois_admission")

 CRÉATION TABLES DELTA LAKE...

 Écriture table : workspace.default.hospital_readmissions_clean
 Table Delta créée : workspace.default.hospital_readmissions_clean

 STATISTIQUES FINALES :
   Lignes : 0
   Colonnes : 35
   Partitions : annee_admission × mois_admission


In [0]:
# ========================================
# OPTIMISATION DELTA LAKE
# ========================================
# WHY : Compaction des petits fichiers + Z-Ordering pour performance
print("OPTIMISATION DELTA TABLE...\n")
# Optimize + Z-Order sur colonnes fréquemment filtrées
spark.sql(f"""
    OPTIMIZE {full_table_name}
    ZORDER BY (patient_id, service, readmission_30j)
""")
print("Optimisation terminée (OPTIMIZE + ZORDER)")


OPTIMISATION DELTA TABLE...

Optimisation terminée (OPTIMIZE + ZORDER)


In [0]:
# ========================================
# RAPPORT FINAL DE CLEANING
# ========================================
print("\n" + "="*70)
print(" RAPPORT FINAL - DATA CLEANING ")
print("="*70 + "\n")
# Charger table finale
df_final = spark.table(full_table_name)
# Métriques clés
print("MÉTRIQUES GLOBALES :")
print(f"Total lignes : {df_final.count():,}")
print(f"Total colonnes : {len(df_final.columns)}")
print(f"Taille estimée (MB) : {df_final.count() * len(df_final.columns) * 8 / 1e6:.2f}")
# Distribution variable cible
print("\nDISTRIBUTION VARIABLE CIBLE :")
target_dist = df_final.groupBy("readmission_30j").count().toPandas()
print(target_dist.to_string(index=False))
imbalance_ratio = target_dist['count'].max() / target_dist['count'].min()
print(f"   Ratio déséquilibre : {imbalance_ratio:.2f}:1")
# Vérification valeurs manquantes finales
print("\n VALEURS MANQUANTES RÉSIDUELLES :")
null_check = []
for col in df_final.columns:
    null_count = df_final.filter(F.col(col).isNull()).count()
    if null_count > 0:
        null_check.append({'colonne': col, 'nb_nulls': null_count})
if null_check:
    print(pd.DataFrame(null_check).to_string(index=False))
else:
    print("Aucune valeur manquante")
# Features créées
print("\nFEATURES CRÉÉES :")
new_features = [
    "categorie_age", "categorie_sejour", "polypathologie",
    "annee_admission", "mois_admission", "jour_semaine", 
    "est_weekend", "ratio_medic_comorb", "duree_sejour_calculee"
]
print(f"   {len(new_features)} nouvelles features")
# Chemins de sortie
print("\nSORTIES :")
print(f"Table Delta : {full_table_name}")
print(f"Format : Delta Lake (partitionné)")
print("\n" + "="*70)
print("DATA CLEANING TERMINÉ - PRÊT POUR FEATURE ENGINEERING")
print("="*70)


 RAPPORT FINAL - DATA CLEANING 

MÉTRIQUES GLOBALES :
Total lignes : 0
Total colonnes : 35
Taille estimée (MB) : 0.00

DISTRIBUTION VARIABLE CIBLE :
Empty DataFrame
Columns: [readmission_30j, count]
Index: []
   Ratio déséquilibre : nan:1

🔍 VALEURS MANQUANTES RÉSIDUELLES :
Aucune valeur manquante

FEATURES CRÉÉES :
   9 nouvelles features

SORTIES :
Table Delta : workspace.default.hospital_readmissions_clean
Format : Delta Lake (partitionné)

DATA CLEANING TERMINÉ - PRÊT POUR FEATURE ENGINEERING


In [0]:
# ========================================
# EXPORT ÉCHANTILLON VALIDATION - VERSION ULTRA-ROBUSTE
# ========================================
print("Export échantillon pour validation manuelle...")
# VÉRIFICATION CRITIQUE : Le DataFrame est-il vide ?
total = df_final.count()
if total == 0:
    print("ERREUR CRITIQUE : df_final est VIDE (0 ligne)")
    print("Vérifiez votre pipeline de nettoyage")
    print("Assurez-vous que df_final = df_clean ou df_raw")
    # Créer un DataFrame vide pour éviter les crashs
    df_sample = df_final.limit(0)
else:
    # Distribution de la variable cible
    count_true = df_final.filter("readmission_30j = true").count()
    count_false = df_final.filter("readmission_30j = false").count()
    print(f"Distribution de la variable cible :")
    print(f"readmission_30j = true  : {count_true} lignes")
    print(f"readmission_30j = false : {count_false} lignes")
    print(f"TOTAL : {total} lignes")
    # Gestion des cas limites
    if count_true == 0 and count_false == 0:
        print("ERREUR : Aucune donnée valide pour readmission_30j")
        df_sample = df_final.limit(0)
    elif count_true == 0:
        print("AVERTISSEMENT : Aucune readmission positive")
        df_sample = df_final.filter("readmission_30j = false").limit(min(500, count_false))
    elif count_false == 0:
        print("AVERTISSEMENT : Aucune readmission négative")
        df_sample = df_final.filter("readmission_30j = true").limit(min(500, count_true))
    else:
        # Échantillonnage stratifié normal
        fraction_true = min(500 / count_true, 1.0)
        fraction_false = min(500 / count_false, 1.0)
        df_sample = df_final.sampleBy(
            "readmission_30j", 
            fractions={
                True: fraction_true,
                False: fraction_false
            }, 
            seed=42
        )
# Export seulement si l'échantillon n'est pas vide
sample_count = df_sample.count()
if sample_count > 0:
    try:
        sample_pd = df_sample.toPandas()
        output_path = "/dbfs/FileStore/hospital_clean_sample.csv"
        sample_pd.to_csv(output_path, index=False)
        print(f"Échantillon sauvegardé ({sample_count} lignes) : {output_path}")
    except Exception as e:
        print(f"ERREUR export : {str(e)}")
else:
    print("Aucun échantillon à exporter")

Export échantillon pour validation manuelle...
ERREUR CRITIQUE : df_final est VIDE (0 ligne)
Vérifiez votre pipeline de nettoyage
Assurez-vous que df_final = df_clean ou df_raw
Aucun échantillon à exporter
